In [12]:
import re
import json
import time
import queue
import threading
import requests
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

In [13]:
DATA_PATH = "Test Dataset/phase3_test.csv"
BASE_URL  = "https://integrate.api.nvidia.com/v1"
 
GPT_MODEL    = "openai/gpt-oss-20b"
GEMMA_MODEL  = "google/gemma-3n-e4b-it"
INVOKE_URL   = "https://integrate.api.nvidia.com/v1/chat/completions" 
 
TEMPERATURE      = 0.0   
RPM              = 30     
MIN_INTERVAL     = 60.0 / RPM   
RATE_LIMIT_SLEEP = 20.0   
RETRY_SLEEP      = 10.0    

In [14]:
API_KEYS = 'YOUR API KEYSS'

In [15]:
SUMMARY_PROMPT = """\
# Instruction
You are a clinical NLP expert and mental health classifier. Your task is to carefully read a person's text message and assess their mental state on a scale of 1 to 10.
Here is the brief description about the dataset:
To classify if the person is struggling with mental health issues or not from their messages.
I will provide you with the user text and a label. Label with 0 means mentally well and score will usually be less than or equal to 5. Label with 1 means not mentally well, which means score would be greater than 5.
This dataset is a collection of texts related to people with anxiety, depression, and other mental health issues. 1 means a comment indicative of mental health issues, 0 means a person who is not struggling.
You should first read the user input carefully, then evaluate based on the Criteria and Rating Rubric below. Give step-by-step reasoning for your rating.
---
# Evaluation
## Criteria
- **Mental Wellbeing**: Overall emotional and psychological state.
- **Hopelessness**: Nihilistic thinking, loss of purpose, belief things cannot improve.
- **Anxiety**: Excessive worry, fear, panic, rumination.
- **Depression**: Persistent sadness, withdrawal, low energy, emotional numbness.
- **Stress**: Overwhelm, burnout, pressure.
- **Suicidal Ideation**: Any direct or indirect references to self-harm or not wanting to live.
- **Groundedness**: Base the assessment only on what the text says.
## Rating Rubric
- 1 (BEST): Excellent mental state. No signs of distress.
- 2 (VERY_GOOD): Mentally well, very minor or no issues.
- 3 (GOOD): Healthy with minor occasional concerns.
- 4 (MODERATE_GOOD): Fairly good, minimal issues.
- 5 (NEUTRAL): Borderline, ambiguous signals.
- 6 (MODERATE_BAD): Moderate signs, monitoring advised.
- 7 (BAD): Significantly compromised, seek help soon.
- 8 (VERY_BAD): Severely compromised, professional help promptly.
- 9 (VERY_VERY_BAD): Very severely compromised, immediate intervention.
- 10 (WORST): Extremely poor, urgent crisis intervention.
## Evaluation Steps
- STEP 1: Identify linguistic signals (hopelessness, anxiety, depression, suicidal ideation, stress, isolation, etc.).
- STEP 2: Cross-reference against the Criteria.
- STEP 3: Assign a score per the Rubric, consistent with the label (label 0 -> score <= 5, label 1 -> score > 5).
- STEP 4: List dominant signals.
- STEP 5: Write a 2-3 sentence reasoning.
---
# Few-Shot Examples
## Score 1 - BEST
**Text:** "Had the most amazing weekend hiking with friends. Feeling so refreshed and grateful for life."
**Label:** 0 -> **Score: 1 - BEST**
## Score 2 - VERY_GOOD
**Text:** "guys finally got a girlfriend after leaving a toxic relationship... my dad said he was proud."
**Label:** 0 -> **Score: 2 - VERY_GOOD**
## Score 3 - GOOD
**Text:** "Work has been a bit stressful lately but I've been managing it fine. Gym helps. My friends have been super supportive."
**Label:** 0 -> **Score: 3 - GOOD**
## Score 4 - MODERATE_GOOD
**Text:** "tell my crush i like her ive been procrastinating for months im still unsure get help pls"
**Label:** 0 -> **Score: 4 - MODERATE_GOOD**
## Score 5 - NEUTRAL
**Text:** "I don't really know how I'm feeling these days. Some days are okay, some days feel really heavy. I'm managing but it's not easy."
**Label:** 0 -> **Score: 5 - NEUTRAL**
## Score 6 - MODERATE_BAD
**Text:** "im tired wish born way care productive wanna connect ppl want cant connect anyone awful feel trapped"
**Label:** 1 -> **Score: 6 - MODERATE_BAD**
## Score 7 - BAD
**Text:** "i've been crying every day for weeks. i can't get out of bed. i've stopped replying to friends. everything feels pointless."
**Label:** 1 -> **Score: 7 - BAD**
## Score 8 - VERY_BAD
**Text:** "dont know how long i can hold on longer... theres nothing going anywhere in my life. its meaningless. cant continue anymore."
**Label:** 1 -> **Score: 8 - VERY_BAD**
## Score 9 - VERY_VERY_BAD
**Text:** "nothing to look forward to in life. dont have many reasons to keep going. makes me want to hang."
**Label:** 1 -> **Score: 9 - VERY_VERY_BAD**
## Score 10 - WORST
**Text:** "cant do this anymore. tried to kill myself twice. wish id succeeded last summer. was hospitalized..."
**Label:** 1 -> **Score: 10 - WORST**
"""
 
OUTPUT_INSTRUCTIONS = """\
---
# Now classify the following
**Text:** {text}
**Label:** {label} ({label_meaning})
# Output format - VERY IMPORTANT
Return ONLY a single valid JSON object on one line. No markdown, no code fences, no prose before or after. Exactly these keys:
{{"score": <integer 1-10>, "rating": "<BEST|VERY_GOOD|GOOD|MODERATE_GOOD|NEUTRAL|MODERATE_BAD|BAD|VERY_BAD|VERY_VERY_BAD|WORST>", "reasoning": "<2-3 sentences>", "dominant_signals": "<comma-separated list>"}}
Remember: label 0 -> score must be 1-5, label 1 -> score must be 6-10.
"""
 
 
def build_user_content(text: str, label: int) -> str:
    meaning = "mentally well - score must be 1-5" if int(label) == 0 else "at risk - score must be 6-10"
    safe = str(text).replace("{", "{{").replace("}", "}}")
    return SUMMARY_PROMPT + OUTPUT_INSTRUCTIONS.format(text=safe, label=int(label), label_meaning=meaning)


In [16]:
def call_gpt(client: OpenAI, key: str, content: str) -> str:
    completion = client.chat.completions.create(
        model=GPT_MODEL,
        messages=[{"role": "user", "content": content}],
        temperature=TEMPERATURE,
        top_p=1,
        max_tokens=4096,
        stream=True,
    )
    parts = []
    for chunk in completion:
        if not getattr(chunk, "choices", None):
            continue
        delta = chunk.choices[0].delta
        if getattr(delta, "content", None) is not None:
            parts.append(delta.content)
    return "".join(parts)


In [17]:
def call_gemma(client: OpenAI, key: str, content: str) -> str:
    headers = {
        "Authorization": f"Bearer {key}",
        "Accept": "text/event-stream",
    }
    payload = {
        "model": GEMMA_MODEL,
        "messages": [{"role": "user", "content": content}],
        "max_tokens": 512,
        "temperature": 0.20,
        "top_p": 0.70,
        "frequency_penalty": 0.00,
        "presence_penalty": 0.00,
        "stream": True,
    }
    parts = []
    with requests.post(INVOKE_URL, headers=headers, json=payload, stream=True, timeout=120) as resp:
        resp.raise_for_status()  
        for raw in resp.iter_lines():
            if not raw:
                continue
            line = raw.decode("utf-8")
            if not line.startswith("data:"):
                continue
            data = line[len("data:"):].strip()
            if data == "[DONE]":
                break
            try:
                obj = json.loads(data)
                piece = obj["choices"][0].get("delta", {}).get("content")
                if piece:
                    parts.append(piece)
            except Exception:
                continue
    return "".join(parts)


In [18]:
def call_llama(client: OpenAI, key: str, content: str) -> str:
    completion = client.chat.completions.create(
        model="meta/llama-3.1-8b-instruct",
        messages=[{"role": "user", "content": content}],
        temperature=0.2,
        top_p=0.7,
        max_tokens=1024,
        stream=True,
    )
    parts = []
    for chunk in completion:
        if not getattr(chunk, "choices", None):
            continue
        delta = chunk.choices[0].delta
        if getattr(delta, "content", None) is not None:
            parts.append(delta.content)
    return "".join(parts)

    if completion.choices[0].message.content is not None:
        print(completion.choices[0].message.content)

In [19]:
def extract_score(raw: str):
    """Return an int score in 1..10, or None if it can't be found."""
    if not raw:
        return None
    txt = re.sub(r"```(?:json)?", "", raw).replace("```", "").strip()
 
    m = re.search(r"\{.*\}", txt, flags=re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            s = int(obj["score"])
            if 1 <= s <= 10:
                return s
        except Exception:
            pass
 
    m = re.search(r'"?score"?\s*[:=]\s*(\d{1,2})', txt, flags=re.IGNORECASE)
    if m:
        s = int(m.group(1))
        if 1 <= s <= 10:
            return s
 
    return None


In [20]:
def score_row(call_fn, client, key, text, label, last_call) -> int:
   
    content = build_user_content(text, label)
    while True:
        wait = MIN_INTERVAL - (time.monotonic() - last_call[0])
        if wait > 0:
            time.sleep(wait)
        last_call[0] = time.monotonic()
 
        try:
            raw = call_fn(client, key, content)
            s = extract_score(raw)
            if s is not None:
                return s
            time.sleep(RETRY_SLEEP)          
        except Exception as e:
            msg = str(e).lower()
            if "429" in msg or "too many requests" in msg or "rate limit" in msg:
                time.sleep(RATE_LIMIT_SLEEP)  
            else:
                time.sleep(RETRY_SLEEP)       

In [21]:
def run_model(call_fn, model_name, keys, out_path):
    df = pd.read_csv(DATA_PATH)
    rows = df.to_dict("records")
    n = len(rows)
    scores = [None] * n            
 
    work = queue.Queue()
    for i in range(n):
        work.put(i)
 
    print(f"[{model_name}] {n} rows | {len(keys)} keys -> {len(keys)} parallel workers")
 
    pbar = tqdm(total=n, desc=model_name, unit="row", smoothing=0.05)
    pbar_lock = threading.Lock()
 
    def worker(key, wid):
        client = OpenAI(base_url=BASE_URL, api_key=key)   
        last_call = [0.0]                               
        while True:
            try:
                idx = work.get_nowait()
            except queue.Empty:
                return
            row = rows[idx]
            s = score_row(call_fn, client, key, row["text"], int(row["label"]), last_call)
            scores[idx] = s
            with pbar_lock:
                pbar.update(1)
            work.task_done()
 
    threads = [threading.Thread(target=worker, args=(k, i), daemon=True)
               for i, k in enumerate(keys)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
 
    pbar.close()
    pd.DataFrame({"score": scores}).to_csv(out_path, index=False)
    print(f"[{model_name}] DONE — {n} scores saved to {out_path}")
 
 
if __name__ == "__main__":
    run_model(call_gemma, "gemma", API_KEYS, "gemma_score.csv")
    run_model(call_gpt,   "gpt",   API_KEYS, "gpt_score.csv")
    run_model(call_llama,   "llama",   API_KEYS, "llama_score.csv")


[gemma] 2798 rows | 31 keys -> 31 parallel workers


gemma: 100%|██████████| 2798/2798 [09:15<00:00,  5.03row/s]


[gemma] DONE — 2798 scores saved to gemma_score.csv
[gpt] 2798 rows | 31 keys -> 31 parallel workers


gpt: 100%|██████████| 2798/2798 [08:11<00:00,  5.69row/s]


[gpt] DONE — 2798 scores saved to gpt_score.csv
[llama] 2798 rows | 31 keys -> 31 parallel workers


llama: 100%|██████████| 2798/2798 [1:45:07<00:00,  2.25s/row]

[llama] DONE — 2798 scores saved to llama_score.csv


## Evaluation

In [48]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    cohen_kappa_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

In [35]:
ORIG_PATH    = "Test Dataset/phase3_test.csv"
GEMMA_SCORES = "gemma_score.csv"
GPT_SCORES   = "gpt_score.csv"
LLAMA_SCORES = "llama_score.csv"

df    = pd.read_csv(ORIG_PATH)
gemma = pd.read_csv(GEMMA_SCORES)["score"].astype(int).reset_index(drop=True)
gpt   = pd.read_csv(GPT_SCORES)["score"].astype(int).reset_index(drop=True)
llama = pd.read_csv(LLAMA_SCORES)["score"].astype(int).reset_index(drop=True)

assert len(df) == len(gemma) == len(gpt), (
    f"Row count mismatch: dataset={len(df)}, gemma={len(gemma)}, gpt={len(gpt)}"
)

gemma_df = df.copy(); gemma_df["pred_score"] = gemma.values
gpt_df   = df.copy(); gpt_df["pred_score"]   = gpt.values
llama_df = df.copy(); llama_df["pred_score"] = llama.values

gemma_df.to_csv("gemma_final_scores.csv", index=False)
gpt_df.to_csv("gpt_final_scores.csv", index=False)
llama_df.to_csv("llama_final_scores.csv", index=False)

print("Saved gemma_final_scores.csv ->", gemma_df.shape)
print("Saved gpt_final_scores.csv   ->", gpt_df.shape)
print("Saved llama_final_scores.csv ->", llama_df.shape)

gemma_df.head()
gpt_df.head()
llama_df.head()


Saved gemma_final_scores.csv -> (2798, 5)
Saved gpt_final_scores.csv   -> (2798, 5)
Saved llama_final_scores.csv -> (2798, 5)


,text,label,qwen_score,class_label,pred_score
0,im tiredi battling anxiety and according one f...,1,9,8,9
1,anxietymy anxiety gets like prison head think ...,1,8,7,9
2,today wake moms phone call crying vacation tel...,1,7,6,9
3,another unreasonable waveif check previous pos...,1,7,6,8
4,one awkward situations happened yesterday eati...,0,4,3,2


In [50]:
import numpy as np
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score, f1_score, mean_absolute_error, 
    cohen_kappa_score, confusion_matrix, classification_report
)

def map_to_5_buckets(scores):
    """
    Collapse the 10-point severity scale into 5 functional buckets 
    as defined in the project methodology.
    
    Scale (1-10): 
    1-2: Healthy, 3-4: Good, 5: Neutral, 6-8: Bad, 9-10: Critical
    """
    s = np.asarray(scores)
    buckets = np.zeros_like(s)
    buckets[(s >= 1) & (s <= 2)] = 0    # Healthy
    buckets[(s >= 3) & (s <= 4)] = 1    # Good
    buckets[s == 5] = 2                 # Neutral
    buckets[(s >= 6) & (s <= 8)] = 3    # Bad
    buckets[(s >= 9) & (s <= 10)] = 4   # Critical
    return buckets

def compute_metrics(labels, preds):
    """
    Calculates ordinal and classification metrics for severity evaluation.
    Expects inputs to be on the 1-10 scale.
    """
    labels = np.asarray(labels)
    preds = np.asarray(preds)

    # Calculate metrics
    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "off_by_one": np.mean(np.abs(preds - labels) <= 1),
        "mae": mean_absolute_error(labels, preds),
        "qwk": cohen_kappa_score(labels, preds, weights="quadratic"),
        "bucket5_acc": accuracy_score(map_to_5_buckets(labels), map_to_5_buckets(preds)),
        # Convert CM to list so it is JSON serializable for the file export
        "confusion_matrix": confusion_matrix(labels, preds, labels=list(range(1, 11))).tolist()
    }
    return metrics


In [51]:
def evaluate_model(name: str, data: pd.DataFrame) -> dict:
    valid = data.dropna(subset=["qwen_score", "pred_score"]).copy()
    valid["qwen_score"] = valid["qwen_score"].astype(int)
    valid["pred_score"] = valid["pred_score"].astype(int)

    METRICS_JSON = f"{name}_metrics.json"
    CM_CSV       = f"{name}_confusion_matrix.csv"

    metrics = compute_metrics(valid["qwen_score"].to_numpy(), valid["pred_score"].to_numpy())

    print("=" * 60)
    print(f"METRICS - {name.upper()}")
    print("=" * 60)
    print(json.dumps({k: v for k, v in metrics.items() if k != "confusion_matrix"}, indent=2))

    cm_df = pd.DataFrame(
        metrics["confusion_matrix"],
        index=[f"true_{i}"  for i in range(1, 11)],
        columns=[f"pred_{i}" for i in range(1, 11)],
    )
    print(f"\nConfusion Matrix (rows = qwen_score truth, cols = {name} pred):")
    print(cm_df)

    print("\nPer-class classification report:")
    print(classification_report(
        valid["qwen_score"], valid["pred_score"],
        labels=list(range(1, 11)), zero_division=0, digits=3,
    ))

    with open(METRICS_JSON, "w") as f:
        json.dump(metrics, f, indent=2)
    cm_df.to_csv(CM_CSV)
    print(f"\nSaved: {METRICS_JSON}, {CM_CSV}")
    return metrics


gemma_metrics = evaluate_model("gemma", gemma_df)
print("\n")
gpt_metrics   = evaluate_model("gpt",   gpt_df)
llama_metrics = evaluate_model("llama", llama_df)

METRICS - GEMMA
{
  "accuracy": 0.4006433166547534,
  "f1_weighted": 0.40659865998534317,
  "off_by_one": 0.7376697641172266,
  "mae": 0.9664045746962115,
  "qwk": 0.9038208452977152,
  "bucket5_acc": 0.643674052894925
}

Confusion Matrix (rows = qwen_score truth, cols = gemma pred):
         pred_1  pred_2  pred_3  pred_4  pred_5  pred_6  pred_7  pred_8  \
true_1      142     110     280      96      17       2       1       0   
true_2       25      46     117      37       7       2       2       0   
true_3        3      11      58      32       7       2       3       0   
true_4        2      14     133      97      15      24      19       0   
true_5        1       5      12      24       8      17      24       1   
true_6        0       2       1       5       6      15     119      24   
true_7        0       0       0       0       0       0      29      19   
true_8        0       0       0       1       0       0      21      19   
true_9        0       0       0       0 

In [52]:
summary = pd.DataFrame({
    "Gemma": {k: v for k, v in gemma_metrics.items() if k != "confusion_matrix"},
    "GPT":   {k: v for k, v in gpt_metrics.items()   if k != "confusion_matrix"},
    "LLAMA": {k: v for k, v in llama_metrics.items() if k != "confusion_matrix"},
})
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
summary

,Gemma,GPT,LLAMA
accuracy,0.4006,0.4825,0.4696
f1_weighted,0.4066,0.5218,0.4624
off_by_one,0.7377,0.8660,0.7881
mae,0.9664,0.6923,0.8742
qwk,0.9038,0.9479,0.9146
bucket5_acc,0.6437,0.7273,0.7312
